# Train Mnx: tools, images, search, code and answers

Fine-tunes **Qwen2.5-3B-Instruct** into your Mnx model on up to **115,000 conversations** (plus what it learned from your chats) that match exactly how the Mnx app talks to the model (`<think>`, `<tool_call>`, `<tool_response>`), then **measures** it on 800 held-out cases and exports GGUF files for your phone.

**Runtime → Change runtime type → GPU.** Then run the cells in order, twice:

1. With `PRESET = "smoke"` (the default). In about 20–30 minutes it trains the real model for 12 steps, stops and resumes from a checkpoint, writes a sample answer, exports both GGUF files, runs 20 eval cases, and **prints how long the full run will take on your GPU**. Nothing is uploaded.
2. With `PRESET = "full"` for the real run.

The data is ~115,000 conversations of ~3,100 tokens each (~355M tokens; most of each is the system prompt, which is read but not learned). Rough times for the full run: several free T4 sessions (progress is saved to Google Drive and resumes), about a day on an L4, about half a day on an A100. Trust the smoke test's estimate over these. Short on time? Set `USE = 25000`.

Using a rented GPU instead of Colab? See *Train on a rented GPU* in the README: it runs the same script.

In [ ]:
# 1. Check the GPU, install Unsloth, connect Google Drive (to keep progress)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth
import os
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/mnx-training"
os.environ["LLAMA_CPP"] = "/content/llama.cpp"   # built once by the export step, reused by the eval

## 1b. Get the Mnx code
If the repo is private, add a Colab secret `GITHUB_TOKEN` (🔑 on the left). Add `HF_TOKEN` (write access) to upload the model at the end.

In [ ]:
import os, subprocess
from google.colab import userdata
BRANCH = "claude/mnx-ai-web-interface-f75vu6"
try:
    gh = userdata.get("GITHUB_TOKEN")
except Exception:
    gh = None
url = f"https://{gh}@github.com/Tharunmaks/Mnx" if gh else "https://github.com/Tharunmaks/Mnx"
if not os.path.exists("Mnx"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "Mnx"], check=True)
!cd Mnx && git pull -q && git log --oneline -1

## 2. Make the data
Generates ~115,000 distinct conversations (the main set plus 15,000 for the 65 quick tools) and 800 held-out test cases, then checks every conversation with Mnx's own parsers.

**Your learned examples** (👍 answers, your corrections, and online-model chats from the Mnx app) are added automatically: run `HF_TOKEN=hf_... npm run learn -- --upload` on your phone first. Or upload `training/data/extra.jsonl` from your phone here as `extra.jsonl`.

In [ ]:
!cd Mnx && npm --version > /dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null)
!cd Mnx && npm install --omit=optional --silent && npm run -s train:data

# What Mnx learned from your chats (uploaded with: npm run learn -- --upload)
import json, os
try:
    from huggingface_hub import hf_hub_download, HfApi
    tok = userdata.get("HF_TOKEN")
    me = HfApi(token=tok).whoami()["name"]
    path = hf_hub_download(f"{me}/mnx-learned", "learned.jsonl", repo_type="dataset", token=tok, force_download=True)
    have = set(open("extra.jsonl").read().splitlines()) if os.path.exists("extra.jsonl") else set()
    with open(path) as src, open("extra.jsonl", "a") as dst:   # re-running this cell adds nothing twice
        dst.writelines(line for line in src if line.strip() and line.rstrip("\n") not in have)
except Exception as e:
    print("No learned examples downloaded:", str(e)[:150])
if os.path.exists("extra.jsonl"):
    print("Learned examples to include:", sum(1 for line in open("extra.jsonl") if line.strip()))

## 3. Train (LoRA), sample answer, export GGUF
Runs `training/train.py`. The full run saves a checkpoint every 250 steps to Google Drive: after a disconnect, re-run cells 1, 1b and this one (skip step 2's data cell only if `Mnx/training/data/train.jsonl` still exists) and it continues where it stopped. Afterwards it exports `mnx-q4_k_m.gguf` (best quality for its size) and `mnx-q4_0.gguf` (faster on phone CPUs) to `SAVE_DIR`.

In [ ]:
PRESET = "smoke"   # "smoke" first; then "full" for the real run
USE = 115000       # full run only: how many conversations (e.g. 25000 for a shorter run)
# Smaller base models answer much faster on phones but are less capable:
#   "unsloth/Qwen2.5-3B-Instruct"   best answers (default)
#   "unsloth/Qwen2.5-1.5B-Instruct" about 2x faster on a phone, trains ~2x faster
#   "unsloth/Qwen2.5-0.5B-Instruct" about 5x faster, noticeably weaker
BASE = "unsloth/Qwen2.5-3B-Instruct"

OUT = f"{SAVE_DIR}/smoke" if PRESET == "smoke" else SAVE_DIR
use = f"--use {USE}" if PRESET == "full" else ""
!python3 Mnx/training/train.py --preset {PRESET} --base {BASE} {use} --out {SAVE_DIR} --work /content/mnx-work --extra extra.jsonl

## 4. Measure accuracy (target 98%)
Runs the held-out cases through Mnx's real tool loop (including the permission step for code) with llama.cpp on the GPU. The smoke run checks only 20 cases (after 12 steps it won't pass yet; this only proves the eval works).

In [ ]:
limit = "--limit 20" if PRESET == "smoke" else ""
!cd Mnx && MNX_LLAMA_SERVER_BIN=/content/llama.cpp/build/bin/llama-server MNX_GPU_LAYERS=99 node training/eval.mjs --model {OUT}/mnx-q4_k_m.gguf --target 98 {limit}

Below 98% somewhere? The failures are printed above and saved in `Mnx/training/data/eval-report.json`. Send them to Claude to improve the data, or train another epoch (add `--epochs 2` to the train command).

## 5. Upload to Hugging Face (full run only)

In [ ]:
from huggingface_hub import HfApi
assert PRESET == "full", "The smoke model is only a test. Set PRESET = \"full\" and train first."
api = HfApi(token=userdata.get("HF_TOKEN"))
repo = "tharunmakes/mnx-qwen2.5-3b-gguf"
api.create_repo(repo, private=True, exist_ok=True)
for f in ["mnx-q4_k_m.gguf", "mnx-q4_0.gguf"]:
    api.upload_file(path_or_fileobj=f"{OUT}/{f}", path_in_repo=f, repo_id=repo)
print("Uploaded. On the phone: HF_TOKEN=hf_... npm run get-model")